# Train our own model

Thin driver: mounts Drive, downloads the dataset, clones the repo, builds the
manifest (reusing `data.coco_split`), then calls `train_model.py` / `export_model.py`
as subprocesses -- no training logic lives in this notebook.

Checkpoints and the exported model are written to Drive (`RUN_ROOT`), same pattern
as the transformer-pytorch repo: `latest.pt` (full resumable state) and `best.pt`
(lightest, only overwritten when validation mAP@0.5 improves).

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

RUN_ROOT = "/content/drive/MyDrive/yolo-edge-quantization"

## 2. Download the dataset

In [ ]:
!pip install -q roboflow

from google.colab import userdata
from roboflow import Roboflow

rf = Roboflow(api_key=userdata.get("ROBOFLOW_API_KEY"))
project = rf.workspace("efe-tatar").project("yolo-to-coco-traffic-signs-pmo2q")
dataset = project.version(1).download("coco")

datasetDirectory = dataset.location
print(datasetDirectory)

## 3. Clone the repo and install its requirements

In [ ]:
import os

repoDirectory = "/content/yolo-edge-quantization"
if not os.path.isdir(repoDirectory):
    !git clone -q https://github.com/efe-tatar/yolo-edge-quantization.git {repoDirectory}
else:
    !git -C {repoDirectory} pull -q

!pip install -q -r {repoDirectory}/requirements.txt

import sys
sys.path.insert(0, repoDirectory)

## 4. Build the manifest

Same `data.coco_split` functions as `dataset_split.ipynb` -- nothing reimplemented
here. Also saves `manifest.json` + `classes.json` to Drive, so this exact split is
on record alongside the checkpoints it produced.

In [ ]:
import json

from data.coco_split import pool_boxes, redistribute, build_manifest

splitNames = ["train", "valid", "test"]
cocoBySplit = {}
for splitName in splitNames:
    with open(os.path.join(datasetDirectory, splitName, "_annotations.coco.json")) as jsonFile:
        cocoBySplit[splitName] = json.load(jsonFile)

boxes = pool_boxes(cocoBySplit)

classNames = sorted({box["class"] for box in boxes})
classToIndex = {className: i for i, className in enumerate(classNames)}
numClasses = len(classNames)
print(numClasses, "classes")

setOfPhoto, trainOnlyClasses = redistribute(boxes, min_photos=4, seed=42)
manifest = build_manifest(boxes, setOfPhoto, classToIndex, datasetDirectory)

for setName in ["train", "valid", "test"]:
    print(setName, ":", len(manifest[setName]), "images")

runDir = f"{RUN_ROOT}/own_model"
os.makedirs(runDir, exist_ok=True)

manifestPath = f"{runDir}/manifest.json"
with open(manifestPath, "w") as f:
    json.dump(manifest, f)

with open(f"{runDir}/classes.json", "w") as f:
    json.dump(classNames, f)

print("saved manifest to", manifestPath)

## 5. Train

`--thresholds 32 80` -- derived from this dataset's real box-size percentiles (p50=29px, p90=81px; see dataset_split.ipynb section 8), not a generic heuristic. Gives stride32 ~10% of boxes instead of ~2%, stride16 ~35% instead of ~15%.
Reruns resume automatically from `runDir/latest.pt`.

In [ ]:
epochs = 50
batchSize = 16

!python {repoDirectory}/train_model.py --manifest {manifestPath} --out-dir {runDir} --num-classes {numClasses} --strides 8 16 32 --thresholds 32 80 --epochs {epochs} --batch-size {batchSize}

## 6. Check the result

In [ ]:
import torch

best = torch.load(f"{runDir}/best.pt", map_location="cpu", weights_only=False)
print("best epoch:", best["epoch"])
print("val mAP@0.5:", best["val_map"])

## 7. Export to ONNX

In [ ]:
onnxPath = f"{runDir}/model.onnx"

!python {repoDirectory}/export_model.py --checkpoint {runDir}/best.pt --num-classes {numClasses} --out {onnxPath}

## 8. Visualize predictions on a handful of test images

A sanity check, not a metric -- this is the first time this whole pipeline has run
on real images rather than synthetic tensors, so it's worth actually looking.
Predictions in red, ground truth in green, on the same image.

In [ ]:
import random

import matplotlib.pyplot as plt
import torchvision.transforms as transforms

from data.dataset import DetectionDataset
from models.yolo import YOLO
from train.decode import decode_predictions
from train.trainer import load_checkpoint
from train.visualize import draw_detections

imgSize = 320  # must match what train_model.py was run with
strides = (8, 16, 32)

with open(f"{runDir}/classes.json") as f:
    classNamesLoaded = json.load(f)

model = YOLO(numClasses)
load_checkpoint(f"{runDir}/best.pt", model, device="cpu")
model.eval()

testDataset = DetectionDataset(manifest["test"], img_size=imgSize)

random.seed(0)
sampleIndices = random.sample(range(len(testDataset)), min(6, len(testDataset)))

figure, axes = plt.subplots(2, 3, figsize=(15, 10))
for axis, idx in zip(axes.flat, sampleIndices):
    imageTensor, gtBoxes, gtLabels = testDataset[idx]

    with torch.no_grad():
        preds = model(imageTensor.unsqueeze(0))
    decoded = decode_predictions(preds, strides, conf_thresh=0.3, iou_thresh=0.5)
    predBoxes, predScores, predLabels = decoded[0]

    image = transforms.ToPILImage()(imageTensor)
    image = draw_detections(image, gtBoxes, gtLabels, class_names=classNamesLoaded, color="green")
    image = draw_detections(image, predBoxes, predLabels, scores=predScores,
                             class_names=classNamesLoaded, color="red")

    axis.imshow(image)
    axis.axis("off")
    axis.set_title(f"{len(predBoxes)} predicted, {len(gtBoxes)} ground truth")

plt.tight_layout()
plt.show()